In [ ]:
import os
import streamlit as st
from datetime import date
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Database configuratie ophalen
load_dotenv()

# Databse gegevens
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DATABASE = os.getenv("DB_NAME")

# SQL connectie
conn_string = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(conn_string)

# Data ophalen
query = "SELECT * FROM training"
df = pd.read_sql(query, con=engine)

# Session state initialiseren
if "trainings" not in st.session_state:
    st.session_state.trainings = []


@st.dialog("Nieuwe training")
def add_training_dialog():
    with st.form("training_form"):
        training_date = st.date_input(
            "Datum",
            value=date.today()
        )

        categorie = st.radio("Training", ["🏋️", "🥋"])
        notitie = st.text_area("Notitie")

        submitted = st.form_submit_button("Opslaan")

        if submitted:
            st.session_state.trainings.append({
                "datum": training_date,
                "categorie": categorie,
                "notitie": notitie
            })

            st.rerun()


# Nieuwe training knop
if st.button("➕ Add new training"):
    add_training_dialog()


st.title("📅 Trainingslogboek")

# Alleen tonen als er trainingen zijn
if st.session_state.trainings:

    # Unieke datums ophalen
    beschikbare_datums = sorted(
        list(
            set(
                training["datum"]
                for training in st.session_state.trainings
            )
        ),
        reverse=True
    )

    geselecteerde_datum = st.selectbox(
        "Selecteer datum",
        beschikbare_datums,
        format_func=lambda x: x.strftime("%d-%m-%Y")
    )

    st.divider()

    # Trainingen van geselecteerde datum tonen
    trainingen_van_datum = [
        training
        for training in st.session_state.trainings
        if training["datum"] == geselecteerde_datum
    ]

    st.subheader(
        f"Trainingen op {geselecteerde_datum.strftime('%d-%m-%Y')}"
    )

    for training in trainingen_van_datum:
        with st.container(border=True):

            col1, col2 = st.columns([4, 1])

            with col1:
                st.markdown(
                    f"### {training['categorie']}"
                )

            with col2:
                st.caption(
                    training["datum"].strftime("%d-%m-%Y")
                )

            st.write(training["notitie"])

else:
    st.info(
        "Nog geen trainingen toegevoegd. Klik op 'Add new training'."
    )